<a href="https://colab.research.google.com/github/HitanshuGedam/quantum-learning-journey/blob/main/Day7_Projective_Measurements_Born_Rule_Collapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7: Projective Measurements - Born Rule and Collapse

## From Quantum States to Classical Outcomes

## Learning Objectives

By the end of this notebook, you will understand:

1. The mathematical formulation of projective measurements
2. The Born rule and how it gives measurement probabilities
3. The collapse postulate (wavefunction collapse)
4. Expectation values of observables
5. How to simulate measurements using QuTiP
6. The difference between projective and general measurements

## Philosophical Motivation

Measurement in quantum mechanics is fundamentally different from classical measurement. The act of measurement not only reveals information but also irreversibly alters the quantum state. This phenomenon — wavefunction collapse — is one of the most debated aspects of quantum foundations. Understanding projective measurements is essential for quantum computing, where measurements extract classical information from quantum states.



## References

- Nielsen & Chuang (2010). Quantum Computation and Quantum Information. Chapter 2.2.
- QuTiP Documentation: https://qutip.org/docs/latest/

In [2]:
# ============================================================================
# SETUP AND INSTALLATIONS
# ============================================================================

!pip install qutip qutip_qip -q

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

print(f"NumPy version: {np.__version__}")
print(f"QuTiP version: {qt.__version__}")
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.8/140.8 kB 13.2 MB/s eta 0:00:00
NumPy version: 2.0.2
QuTiP version: 5.2.3
Setup complete.


## 1. Projective Measurements

### Definition

A projective measurement (also called a von Neumann measurement) is defined by a set of **projection operators** $\{P_m\}$ satisfying:

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Property</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Mathematical Condition</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Orthogonality</td>
            <td style="padding: 8px;">$P_m P_n = \delta_{mn} P_m$</td>
            <td style="padding: 8px;">Different outcomes correspond to orthogonal subspaces</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Completeness</td>
            <td style="padding: 8px;">$\sum_m P_m = I$</td>
            <td style="padding: 8px;">Probabilities sum to 1</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Idempotence</td>
            <td style="padding: 8px;">$P_m^2 = P_m$</td>
            <td style="padding: 8px;">Repeated measurement gives same result</td>
        </tr>
    </tbody>
</table>

### Computational Basis Measurement

For a single qubit, measurement in the computational basis uses:

$$ P_0 = |0\rangle\langle 0| = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}, \quad P_1 = |1\rangle\langle 1| = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} $$

### Observable Formalism

Any projective measurement corresponds to measuring an **observable** (Hermitian operator):

$$ M = \sum_m m P_m $$

where $m$ are the measurement outcomes (eigenvalues).

In [3]:
# ============================================================================
# PROJECTIVE MEASUREMENT OPERATORS
# ============================================================================

print("=" * 70)
print("PROJECTIVE MEASUREMENT OPERATORS")
print("=" * 70)

# Computational basis projectors
P0 = qt.basis(2, 0) * qt.basis(2, 0).dag()
P1 = qt.basis(2, 1) * qt.basis(2, 1).dag()

print("\nComputational basis projectors:")
print(f"\nP0 = |0⟩⟨0| =")
print(P0)
print(f"\nP1 = |1⟩⟨1| =")
print(P1)

# Verify properties
print("\n" + "=" * 50)
print("VERIFYING PROJECTOR PROPERTIES")
print("=" * 50)

# Orthogonality
print(f"\nP0 * P1 = 0? {np.allclose((P0 * P1).full(), np.zeros((2, 2)))}")

# Idempotence
print(f"P0² = P0? {np.allclose((P0 * P0).full(), P0.full())}")
print(f"P1² = P1? {np.allclose((P1 * P1).full(), P1.full())}")

# Completeness
I = qt.qeye(2)
print(f"P0 + P1 = I? {np.allclose((P0 + P1).full(), I.full())}")

# Observables
print("\n" + "=" * 50)
print("OBSERVABLES")
print("=" * 50)

# Pauli Z observable (eigenvalues ±1)
Z = qt.sigmaz()
print(f"\nPauli Z (computational basis observable):")
print(Z)

# Pauli X observable
X = qt.sigmax()
print(f"\nPauli X (X-basis observable):")
print(X)

# Verify eigenvalues
print(f"\nEigenvalues of Z: {Z.eigenenergies()}")
print(f"Eigenvalues of X: {X.eigenenergies()}")

PROJECTIVE MEASUREMENT OPERATORS

Computational basis projectors:

P0 = |0⟩⟨0| =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]

P1 = |1⟩⟨1| =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0.]
 [0. 1.]]

VERIFYING PROJECTOR PROPERTIES

P0 * P1 = 0? True
P0² = P0? True
P1² = P1? True
P0 + P1 = I? True

OBSERVABLES

Pauli Z (computational basis observable):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 1.  0.]
 [ 0. -1.]]

Pauli X (X-basis observable):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 1.]
 [1. 0.]]

Eigenvalues of Z: [-1.  1.]
Eigenvalues of X: [-1.  1.]


## 2. The Born Rule

### Statement

For a quantum state $|\psi\rangle$, the probability of obtaining outcome $m$ when measuring the observable $M = \sum_m m P_m$ is:

$$ p(m) = \langle\psi|P_m|\psi\rangle $$

### For a Qubit in Computational Basis

For a state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$:

$$ p(0) = |\alpha|^2, \quad p(1) = |\beta|^2 $$

### For Density Matrices

For a mixed state $\rho$:

$$ p(m) = \text{Tr}(P_m \rho) $$

### Expectation Value

The expectation value of an observable $M$ is:

$$ \langle M \rangle = \langle\psi|M|\psi\rangle = \sum_m m \, p(m) $$

### Key Properties of the Born Rule

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Property</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Non-negativity</td>
            <td style="padding: 8px;">$p(m) \geq 0$ (since projectors are positive)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Normalization</td>
            <td style="padding: 8px;">$\sum_m p(m) = \text{Tr}(\rho \sum_m P_m) = \text{Tr}(\rho) = 1$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Linearity</td>
            <td style="padding: 8px;">Probability is linear in the state</td>
        </tr>
    </tbody>
</table>

In [4]:
# ============================================================================
# BORN RULE DEMONSTRATION
# ============================================================================

print("=" * 70)
print("BORN RULE DEMONSTRATION")
print("=" * 70)

# Define a superposition state
alpha = np.sqrt(0.3)  # |α|² = 0.3
beta = np.sqrt(0.7)   # |β|² = 0.7
psi = alpha * qt.basis(2, 0) + beta * qt.basis(2, 1)
psi = psi.unit()

print(f"\nState |ψ⟩ = {psi}")
print(f"  |α|² = {abs(alpha)**2:.3f}")
print(f"  |β|² = {abs(beta)**2:.3f}")

# Projectors
P0 = qt.basis(2, 0) * qt.basis(2, 0).dag()
P1 = qt.basis(2, 1) * qt.basis(2, 1).dag()

# Compute probabilities using Born rule
prob_0 = (psi.dag() * P0 * psi).real
prob_1 = (psi.dag() * P1 * psi).real

print(f"\nBorn rule probabilities:")
print(f"  p(0) = ⟨ψ|P0|ψ⟩ = {prob_0:.3f}")
print(f"  p(1) = ⟨ψ|P1|ψ⟩ = {prob_1:.3f}")
print(f"  Sum = {prob_0 + prob_1:.3f}")

# For density matrix
rho = psi * psi.dag()
prob_0_dm = (P0 * rho).tr().real
prob_1_dm = (P1 * rho).tr().real

print(f"\nUsing density matrix: Tr(Pρ)")
print(f"  p(0) = Tr(P0 ρ) = {prob_0_dm:.3f}")
print(f"  p(1) = Tr(P1 ρ) = {prob_1_dm:.3f}")

# Expectation value of Pauli Z
Z = qt.sigmaz()
exp_z = (psi.dag() * Z * psi).real
print(f"\nExpectation value ⟨Z⟩ = {exp_z:.3f}")
print(f"  Alternative: (+1)*p(0) + (-1)*p(1) = {1*prob_0 + (-1)*prob_1:.3f}")

print("\n✅ Born rule correctly gives measurement probabilities!")

BORN RULE DEMONSTRATION

State |ψ⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.54772256]
 [0.83666003]]
  |α|² = 0.300
  |β|² = 0.700

Born rule probabilities:
  p(0) = ⟨ψ|P0|ψ⟩ = 0.300
  p(1) = ⟨ψ|P1|ψ⟩ = 0.700
  Sum = 1.000

Using density matrix: Tr(Pρ)
  p(0) = Tr(P0 ρ) = 0.300
  p(1) = Tr(P1 ρ) = 0.700

Expectation value ⟨Z⟩ = -0.400
  Alternative: (+1)*p(0) + (-1)*p(1) = -0.400

✅ Born rule correctly gives measurement probabilities!


## 3. The Collapse Postulate

### Statement

After a measurement yields outcome $m$, the quantum state collapses to:

$$ |\psi'\rangle = \frac{P_m |\psi\rangle}{\sqrt{p(m)}} $$

For density matrices:

$$ \rho' = \frac{P_m \rho P_m}{p(m)} $$

### Key Features

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Feature</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Irreversibility</td>
            <td style="padding: 8px;">Collapse is irreversible (unlike unitary evolution)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Randomness</td>
            <td style="padding: 8px;">Outcome is probabilistic, not deterministic</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Instantaneity</td>
            <td style="padding: 8px;">Collapse happens instantaneously across the system</td>
        </tr>
    </tbody>
</table>

### Example: Measuring $|+\rangle$ in Computational Basis

For $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$:

- $p(0) = 1/2$, collapse to $|0\rangle$
- $p(1) = 1/2$, collapse to $|1\rangle$

### The Measurement Problem

The collapse postulate is controversial because:
- It introduces non-unitary evolution into quantum mechanics
- The boundary between quantum and classical is not well-defined
- Different interpretations (Copenhagen, Many-Worlds, Bohmian) treat collapse differently

For practical quantum computing, we accept the postulate as it successfully predicts experimental outcomes.

In [5]:
# ============================================================================
# COLLAPSE DEMONSTRATION
# ============================================================================

print("=" * 70)
print("COLLAPSE DEMONSTRATION")
print("=" * 70)

# Create |+⟩ state
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)
psi_plus = (ket0 + ket1).unit()

print(f"\nInitial state: |+⟩ = {psi_plus}")
print(f"  |+⟩ = (|0⟩ + |1⟩)/√2")

# Projectors
P0 = ket0 * ket0.dag()
P1 = ket1 * ket1.dag()

# Probabilities
prob_0 = (psi_plus.dag() * P0 * psi_plus).real
prob_1 = (psi_plus.dag() * P1 * psi_plus).real

print(f"\nProbabilities:")
print(f"  p(0) = {prob_0:.3f}")
print(f"  p(1) = {prob_1:.3f}")

# Collapse upon measuring 0
collapsed_0 = (P0 * psi_plus) / np.sqrt(prob_0)
print(f"\nIf outcome = 0, collapsed state: |ψ'⟩ = {collapsed_0}")
print(f"  Expected: |0⟩ = {ket0}")

# Collapse upon measuring 1
collapsed_1 = (P1 * psi_plus) / np.sqrt(prob_1)
print(f"\nIf outcome = 1, collapsed state: |ψ'⟩ = {collapsed_1}")
print(f"  Expected: |1⟩ = {ket1}")

# Verify normalization
print(f"\nNorm after collapse (outcome 0): {collapsed_0.norm():.3f}")
print(f"Norm after collapse (outcome 1): {collapsed_1.norm():.3f}")

print("\n✅ Collapse correctly projects onto the measurement outcome!")

COLLAPSE DEMONSTRATION

Initial state: |+⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678]
 [0.70710678]]
  |+⟩ = (|0⟩ + |1⟩)/√2

Probabilities:
  p(0) = 0.500
  p(1) = 0.500

If outcome = 0, collapsed state: |ψ'⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]
  Expected: |0⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]

If outcome = 1, collapsed state: |ψ'⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]]
  Expected: |1⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]]

Norm after collapse (outcome 0): 1.000
Norm after collapse (outcome 1): 1.000

✅ Collapse correctly projects onto the measurement outcome!


## 4. Simulating Measurements with QuTiP

### Single Measurement

QuTiP provides the `measurement.measure` function for simulating projective measurements.

### Syntax

```python
from qutip.measurement import measure, measurement_statistics

# Single measurement
result, state = measure(state, op, targets=None)

In [12]:
# ============================================================================
# SIMULATING MEASUREMENTS WITH QuTiP
# ============================================================================

from qutip.measurement import measure, measurement_statistics

print("=" * 70)
print("SIMULATING MEASUREMENTS WITH QuTiP")
print("=" * 70)

# Create a superposition state
alpha = np.sqrt(0.3)
beta = np.sqrt(0.7)
psi = alpha * qt.basis(2, 0) + beta * qt.basis(2, 1)
psi = psi.unit()

print(f"\nState: |ψ⟩ = {psi}")
print(f"  |α|² = {abs(alpha)**2:.3f}")
print(f"  |β|² = {abs(beta)**2:.3f}")

# ============================================================================
# SINGLE MEASUREMENT
# ============================================================================
print("\n" + "=" * 50)
print("SINGLE MEASUREMENT")
print("=" * 50)

# Perform a single measurement in computational basis (Pauli Z)
result, collapsed_state = measure(psi, qt.sigmaz())

# result is the eigenvalue (float)
print(f"\nMeasurement eigenvalue: {result}")
print(f"Collapsed state: {collapsed_state}")

# ============================================================================
# MEASUREMENT STATISTICS
# ============================================================================
print("\n" + "=" * 50)
print("MEASUREMENT STATISTICS")
print("=" * 50)

# Get all possible outcomes, probabilities, and collapsed states
outcomes, probabilities, states = measurement_statistics(psi, qt.sigmaz())

print("\nAll possible measurement outcomes:")
for outcome, prob, state in zip(outcomes, probabilities, states):
    # outcome is the eigenvalue (float)
    # prob is a Qobj, need to extract real value
    prob_value = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
    if outcome == 1:
        bit_result = 0
    else:
        bit_result = 1
    print(f"  Outcome (bit): {bit_result}, Probability: {prob_value:.3f}")
    print(f"    Collapsed state: {state}")

print(f"\nSum of probabilities: {sum([p.full().real[0, 0] for p in probabilities]):.3f}")

# ============================================================================
# MULTIPLE SIMULATED MEASUREMENTS (MONTE CARLO)
# ============================================================================
print("\n" + "=" * 50)
print("MULTIPLE SIMULATED MEASUREMENTS (1000 runs)")
print("=" * 50)

num_runs = 1000
outcomes_list = []

for _ in range(num_runs):
    result, _ = measure(psi, qt.sigmaz())
    # result is the eigenvalue (float)
    if result == 1:
        outcomes_list.append(0)
    else:
        outcomes_list.append(1)

# Count frequencies
unique, counts = np.unique(outcomes_list, return_counts=True)
frequencies = counts / num_runs

print(f"\nSimulated frequencies ({num_runs} runs):")
for outcome, freq in zip(unique, frequencies):
    print(f"  Outcome {outcome}: {freq:.3f}")

print(f"\nTheoretical probabilities:")
print(f"  Outcome 0: {abs(alpha)**2:.3f}")
print(f"  Outcome 1: {abs(beta)**2:.3f}")

# ============================================================================
# MEASUREMENT IN X-BASIS
# ============================================================================
print("\n" + "=" * 50)
print("MEASUREMENT IN X-BASIS")
print("=" * 50)

# Pauli X has eigenvectors |+⟩ (eigenvalue +1) and |-⟩ (eigenvalue -1)
result_x, collapsed_state_x = measure(psi, qt.sigmax())

print(f"\nX-basis measurement eigenvalue: {result_x}")
print(f"Collapsed state: {collapsed_state_x}")

# Statistics for X-basis
outcomes_x, probabilities_x, states_x = measurement_statistics(psi, qt.sigmax())

print("\nX-basis measurement statistics:")
for outcome, prob, state in zip(outcomes_x, probabilities_x, states_x):
    prob_value = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
    print(f"  Eigenvalue: {outcome:.0f}, Probability: {prob_value:.3f}")
    print(f"    Collapsed state: {state}")

# ============================================================================
# MEASUREMENT ON A MIXED STATE
# ============================================================================
print("\n" + "=" * 50)
print("MEASUREMENT ON A MIXED STATE")
print("=" * 50)

# Create a mixed state: 75% |0⟩, 25% |1⟩
rho_mixed = 0.75 * (qt.basis(2, 0) * qt.basis(2, 0).dag()) + \
            0.25 * (qt.basis(2, 1) * qt.basis(2, 1).dag())

print(f"Mixed state ρ = 0.75|0⟩⟨0| + 0.25|1⟩⟨1|")
print(f"ρ = {rho_mixed}")

# Measurement statistics for mixed state
outcomes_mix, probs_mix, states_mix = measurement_statistics(rho_mixed, qt.sigmaz())

print("\nMeasurement statistics for mixed state:")
for outcome, prob, state in zip(outcomes_mix, probs_mix, states_mix):
    prob_value = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
    if outcome == 1:
        bit_result = 0
    else:
        bit_result = 1
    print(f"  Outcome {bit_result}: probability = {prob_value:.3f}")

print("\n✅ QuTiP correctly simulates measurements on both pure and mixed states!")

SIMULATING MEASUREMENTS WITH QuTiP

State: |ψ⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.54772256]
 [0.83666003]]
  |α|² = 0.300
  |β|² = 0.700

SINGLE MEASUREMENT

Measurement eigenvalue: -1.0
Collapsed state: Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]]

MEASUREMENT STATISTICS

All possible measurement outcomes:
  Outcome (bit): 1, Probability: 0.000
    Collapsed state: 0.7000000000000001
  Outcome (bit): 0, Probability: 1.000
    Collapsed state: 0.29999999999999993

Sum of probabilities: 1.000

MULTIPLE SIMULATED MEASUREMENTS (1000 runs)

Simulated frequencies (1000 runs):
  Outcome 0: 0.295
  Outcome 1: 0.705

Theoretical probabilities:
  Outcome 0: 0.300
  Outcome 1: 0.700

MEASUREMENT IN X-BASIS

X-basis measurement eigenvalue: 1.0
Collapsed state: Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678]
 [0.70710678]]

X-basis measurement st

## 5. Expectation Values

### Definition

The expectation value of an observable $M$ in a state $|\psi\rangle$ is:

$$ \langle M \rangle = \langle\psi|M|\psi\rangle $$

### Physical Interpretation

- Average value of many measurements on identically prepared systems
- Not necessarily equal to any single measurement outcome
- Can be calculated without simulating individual measurements

### For Density Matrices

$$ \langle M \rangle = \text{Tr}(M \rho) $$

### Properties

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Property</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Formula</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Linearity</td>
            <td style="padding: 8px;">$\langle aA + bB \rangle = a\langle A \rangle + b\langle B \rangle$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Reality</td>
            <td style="padding: 8px;">$\langle M \rangle$ is real for Hermitian $M$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Born rule connection</td>
            <td style="padding: 8px;">$\langle M \rangle = \sum_m m \, p(m)$</td>
        </tr>
    </tbody>
</table>



In [13]:
# ============================================================================
# EXPECTATION VALUES DEMONSTRATION
# ============================================================================

from qutip import expect

print("=" * 70)
print("EXPECTATION VALUES")
print("=" * 70)

# ============================================================================
# EXPECTATION VALUES FOR PURE STATES
# ============================================================================
print("\n" + "=" * 50)
print("PURE STATE EXPECTATION VALUES")
print("=" * 50)

# Create a state on the equator (|+⟩)
psi_plus = (qt.basis(2, 0) + qt.basis(2, 1)).unit()
print(f"\nState: |+⟩ = {psi_plus}")

# Pauli matrices
X = qt.sigmax()
Y = qt.sigmay()
Z = qt.sigmaz()

# Method 1: Direct calculation
exp_x_direct = (psi_plus.dag() * X * psi_plus).real
exp_y_direct = (psi_plus.dag() * Y * psi_plus).real
exp_z_direct = (psi_plus.dag() * Z * psi_plus).real

print(f"\nDirect calculation (⟨ψ|M|ψ⟩):")
print(f"  ⟨X⟩ = {exp_x_direct:.3f}")
print(f"  ⟨Y⟩ = {exp_y_direct:.3f}")
print(f"  ⟨Z⟩ = {exp_z_direct:.3f}")

# Method 2: Using expect function
exp_x_expect = expect(X, psi_plus)
exp_y_expect = expect(Y, psi_plus)
exp_z_expect = expect(Z, psi_plus)

print(f"\nUsing expect() function:")
print(f"  ⟨X⟩ = {exp_x_expect:.3f}")
print(f"  ⟨Y⟩ = {exp_y_expect:.3f}")
print(f"  ⟨Z⟩ = {exp_z_expect:.3f}")

# ============================================================================
# EXPECTATION VALUES FOR DIFFERENT STATES
# ============================================================================
print("\n" + "=" * 50)
print("EXPECTATION VALUES FOR DIFFERENT STATES")
print("=" * 50)

# State on Y-axis (|+i⟩)
psi_y = (qt.basis(2, 0) + 1j * qt.basis(2, 1)).unit()
print(f"\nState: |+i⟩ = {psi_y}")

exp_x = expect(X, psi_y)
exp_y = expect(Y, psi_y)
exp_z = expect(Z, psi_y)

print(f"  ⟨X⟩ = {exp_x:.3f}")
print(f"  ⟨Y⟩ = {exp_y:.3f}")
print(f"  ⟨Z⟩ = {exp_z:.3f}")

# State |0⟩ (north pole)
psi_0 = qt.basis(2, 0)
print(f"\nState: |0⟩ = {psi_0}")

exp_x = expect(X, psi_0)
exp_y = expect(Y, psi_0)
exp_z = expect(Z, psi_0)

print(f"  ⟨X⟩ = {exp_x:.3f}")
print(f"  ⟨Y⟩ = {exp_y:.3f}")
print(f"  ⟨Z⟩ = {exp_z:.3f}")

# ============================================================================
# EXPECTATION VALUES FOR MIXED STATES
# ============================================================================
print("\n" + "=" * 50)
print("MIXED STATE EXPECTATION VALUES")
print("=" * 50)

# Create a mixed state: 75% |0⟩, 25% |1⟩
rho_mixed = 0.75 * (qt.basis(2, 0) * qt.basis(2, 0).dag()) + \
            0.25 * (qt.basis(2, 1) * qt.basis(2, 1).dag())

print(f"\nMixed state: ρ = 0.75|0⟩⟨0| + 0.25|1⟩⟨1|")
print(f"ρ = {rho_mixed}")

# Calculate expectation values using trace
exp_x = (X * rho_mixed).tr().real
exp_y = (Y * rho_mixed).tr().real
exp_z = (Z * rho_mixed).tr().real

print(f"\nExpectation values using Tr(Mρ):")
print(f"  ⟨X⟩ = {exp_x:.3f}")
print(f"  ⟨Y⟩ = {exp_y:.3f}")
print(f"  ⟨Z⟩ = {exp_z:.3f}")

# Alternative: Using expect with density matrix
exp_x_alt = expect(X, rho_mixed)
exp_y_alt = expect(Y, rho_mixed)
exp_z_alt = expect(Z, rho_mixed)

print(f"\nUsing expect() with density matrix:")
print(f"  ⟨X⟩ = {exp_x_alt:.3f}")
print(f"  ⟨Y⟩ = {exp_y_alt:.3f}")
print(f"  ⟨Z⟩ = {exp_z_alt:.3f}")

# ============================================================================
# VERIFICATION: ⟨Z⟩ = p(0) - p(1)
# ============================================================================
print("\n" + "=" * 50)
print("VERIFICATION: ⟨Z⟩ = p(0) - p(1)")
print("=" * 50)

# For the mixed state
prob_0 = 0.75
prob_1 = 0.25
p0_minus_p1 = prob_0 - prob_1

print(f"\nFor mixed state (75% |0⟩, 25% |1⟩):")
print(f"  p(0) - p(1) = {prob_0:.3f} - {prob_1:.3f} = {p0_minus_p1:.3f}")
print(f"  ⟨Z⟩ = {exp_z:.3f}")
print(f"  Match? {np.isclose(exp_z, p0_minus_p1)}")

# For |+⟩ state
prob_0_plus = 0.5
prob_1_plus = 0.5
p0_minus_p1_plus = prob_0_plus - prob_1_plus

print(f"\nFor |+⟩ state (50% |0⟩, 50% |1⟩):")
print(f"  p(0) - p(1) = {prob_0_plus:.3f} - {prob_1_plus:.3f} = {p0_minus_p1_plus:.3f}")
print(f"  ⟨Z⟩ = {exp_z_expect:.3f}")
print(f"  Match? {np.isclose(exp_z_expect, p0_minus_p1_plus)}")

# ============================================================================
# EXPECTATION VALUES FOR MULTIPLE STATES
# ============================================================================
print("\n" + "=" * 50)
print("EXPECTATION VALUES FOR MULTIPLE STATES")
print("=" * 50)

# Create a list of states
states_list = [psi_0, psi_plus, psi_y]
state_names = ["|0⟩", "|+⟩", "|+i⟩"]

print("\nExpectation values for multiple states:")
print(f"\n{'State':<10} {'⟨X⟩':<10} {'⟨Y⟩':<10} {'⟨Z⟩':<10}")
print("-" * 40)

for name, state in zip(state_names, states_list):
    ex = expect(X, state)
    ey = expect(Y, state)
    ez = expect(Z, state)
    print(f"{name:<10} {ex:+.3f}      {ey:+.3f}      {ez:+.3f}")

print("\n✅ Expectation values correctly calculated using multiple methods!")

EXPECTATION VALUES

PURE STATE EXPECTATION VALUES

State: |+⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678]
 [0.70710678]]

Direct calculation (⟨ψ|M|ψ⟩):
  ⟨X⟩ = 1.000
  ⟨Y⟩ = 0.000
  ⟨Z⟩ = 0.000

Using expect() function:
  ⟨X⟩ = 1.000
  ⟨Y⟩ = 0.000
  ⟨Z⟩ = 0.000

EXPECTATION VALUES FOR DIFFERENT STATES

State: |+i⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678+0.j        ]
 [0.        +0.70710678j]]
  ⟨X⟩ = 0.000
  ⟨Y⟩ = 1.000
  ⟨Z⟩ = 0.000

State: |0⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]
  ⟨X⟩ = 0.000
  ⟨Y⟩ = 0.000
  ⟨Z⟩ = 1.000

MIXED STATE EXPECTATION VALUES

Mixed state: ρ = 0.75|0⟩⟨0| + 0.25|1⟩⟨1|
ρ = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.75 0.  ]
 [0.   0.25]]

Expectation values using Tr(Mρ):
  ⟨X⟩ = 0.000
  ⟨Y⟩ = 0.000
  ⟨Z⟩ = 0.500

Using expect() with 

## 6. Summary and Key Insights

### Projective Measurement Summary

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Concept</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Formula</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Projection Operators</td>
            <td style="padding: 8px;">$P_m = |m\rangle\langle m|$, $P_m P_n = \delta_{mn} P_m$, $\sum_m P_m = I$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Born Rule (Probability)</td>
            <td style="padding: 8px;">$p(m) = \langle\psi|P_m|\psi\rangle = \text{Tr}(P_m \rho)$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Collapse (Pure State)</td>
            <td style="padding: 8px;">$|\psi'\rangle = \frac{P_m|\psi\rangle}{\sqrt{p(m)}}$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Collapse (Mixed State)</td>
            <td style="padding: 8px;">$\rho' = \frac{P_m \rho P_m}{p(m)}$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Expectation Value</td>
            <td style="padding: 8px;">$\langle M \rangle = \langle\psi|M|\psi\rangle = \text{Tr}(M\rho) = \sum_m m \, p(m)$</td>
        </tr>
    </tbody>
</table>

### QuTiP Measurement Functions

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Function</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Returns</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">`measure(state, operator)`</td>
            <td style="padding: 8px;">Single measurement</td>
            <td style="padding: 8px;">(eigenvalue, collapsed_state)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">`measurement_statistics(state, operator)`</td>
            <td style="padding: 8px;">All possible outcomes</td>
            <td style="padding: 8px;">(outcomes, probabilities, states)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">`expect(operator, state)`</td>
            <td style="padding: 8px;">Expectation value</td>
            <td style="padding: 8px;">float</td>
        </tr>
    </tbody>
</table>

### Key Properties of Projective Measurements

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Property</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Orthogonality</td>
            <td style="padding: 8px;">Different outcomes correspond to orthogonal subspaces</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Completeness</td>
            <td style="padding: 8px;">Probabilities sum to 1</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Idempotence</td>
            <td style="padding: 8px;">$P_m^2 = P_m$ (repeated measurement gives same result)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Irreversibility</td>
            <td style="padding: 8px;">Collapse is non-unitary</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Randomness</td>
            <td style="padding: 8px;">Outcome is probabilistic, not deterministic</td>
        </tr>
    </tbody>
</table>

### Key Takeaways

1. **Projective measurements** are defined by orthogonal projectors $\{P_m\}$ with $\sum_m P_m = I$
2. **Born rule** gives measurement probabilities: $p(m) = \langle\psi|P_m|\psi\rangle$
3. **Collapse postulate** describes state update after measurement
4. **Expectation values** give average measurement outcomes: $\langle M \rangle = \sum_m m \, p(m)$
5. **Measurements are probabilistic and irreversible** — fundamentally different from classical measurements



**Day 7 Complete!** 🎉

You now understand:
- The mathematical formulation of projective measurements
- The Born rule and how it gives measurement probabilities
- The collapse postulate (wavefunction collapse)
- Expectation values of observables
- How to simulate measurements using QuTiP's `measure()`, `measurement_statistics()`, and `expect()`

Proceed to Day 8 when ready.